# G1 Academy 2 - Solution: state, modes, and locomotion


## Introduction
Use native subscribers to observe state and native LocoClient, MotionSwitcherClient, and RobotStateClient for modes/services. util.FSM_IDS provides documented mapping data only; participants make the client calls.

## How to reason about direct state and control

A subscriber callback runs asynchronously, so a cached message can be absent or stale even though the subscription object exists. Record receipt time in the callback and refuse motion when required state is old. LocoClient performs locomotion commands; MotionSwitcherClient identifies/releases the higher-level controller; RobotStateClient lists and switches named services. Those APIs are separate: changing an FSM state does not automatically resolve a competing controller or stale publisher.

Use documented mapping data from util.FSM_IDS, but inspect return codes and observed state after every request. The solution demonstrates structure, not permission to activate every mode.


## Task 1 - Subscribe and normalize state
Imports: SportModeState_ is odom-mode state; String_ carries SLAM text. Build callback caches and return simple dictionaries, checking timestamps.


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelSubscriber
from unitree_sdk2py.idl.unitree_go.msg.dds_ import SportModeState_
from unitree_sdk2py.idl.std_msgs.msg.dds_ import String_
ChannelFactoryInitialize(0, "eth0")
class Latest:
    def __init__(self, topic, typ):
        self.msg=None; self.ts=0.0; self.sub=ChannelSubscriber(topic, typ); self.sub.Init(self.cb, 10)
    def cb(self, msg): self.msg=msg; self.ts=time.time()
odom_sub=Latest("rt/odommodestate", SportModeState_); slam_sub=Latest("rt/slam_info", String_)
def observe_state():
    return {"odom": odom_sub.msg, "odom_age_s": time.time()-odom_sub.ts, "slam": None if slam_sub.msg is None else slam_sub.msg.data}
# print(observe_state())


## Task 2 - Use native mode/service clients
LocoClient changes locomotion state; MotionSwitcherClient checks/releases the higher-level motion owner; RobotStateClient lists/switches services. Do not invent FSM IDs: use util.FSM_IDS.


In [ ]:
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
from unitree_sdk2py.comm.motion_switcher.motion_switcher_client import MotionSwitcherClient
try:
    from unitree_sdk2py.b2.robot_state.robot_state_client import RobotStateClient
except ImportError:
    from unitree_sdk2py.go2.robot_state.robot_state_client import RobotStateClient
from util import FSM_IDS
loco=LocoClient(); loco.SetTimeout(5.0); loco.Init()
switcher=MotionSwitcherClient(); switcher.SetTimeout(5.0); switcher.Init()
services=RobotStateClient(); services.SetTimeout(5.0); services.Init()
def set_mode(name): return loco.SetFsmId(FSM_IDS[name])
def toggle_service(name):
    code, rows = services.ServiceList()
    row = next(item for item in rows if item.name == name)
    return services.ServiceSwitch(row.name, not bool(row.status))
# set_mode("damp")


## Task 3 - Issue bounded locomotion
LocoClient.Move accepts velocity commands. Stop with StopMove in finally. Non-continuous Move is an SDK request; observe odometry before/after and do not treat its return code as arrival proof.


In [ ]:
import time
def loco_move(vx, vy, yaw_rate, duration_s=0.5):
    loco.Move(vx, vy, yaw_rate, continous_move=True)
    try: time.sleep(duration_s)
    finally: loco.StopMove()
def odom_move(dx, dy, dyaw):
    before=observe_state(); code=loco.Move(dx, dy, dyaw, continous_move=False)
    return {"before": before, "code": code, "after": observe_state()}
# loco_move(0.03, 0, 0)


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
